# STEP 4 QLoRA Training - Drive Safe Final

이 노트북은 Colab T4 기준 최종 실행본입니다.

핵심 원칙:
- 학습 결과는 `/content`가 아니라 Google Drive에 저장합니다.
- checkpoint를 Drive에 저장하므로 세션이 끊겨도 이어서 학습할 수 있습니다.
- `PY` heredoc을 쓰지 않습니다.
- repo 구조는 `YEH1230/Capstone` 루트에 스크립트, `step4/`에 데이터가 있는 기준입니다.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. Google Drive 마운트 및 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/YEH1230/Capstone.git"
REPO_DIR = "/content/Capstone"

DRIVE_ROOT = "/content/drive/MyDrive/step4_qwen_runs"
OUTPUT_DIR = f"{DRIVE_ROOT}/qwen_step4_lora_fast"
REPORT_DIR = f"{DRIVE_ROOT}/reports_fast"
HF_REPO_ID = "bbanany/qwen2.5-1.5b-step4-pii-qlora"

!mkdir -p {DRIVE_ROOT}
print("OUTPUT_DIR =", OUTPUT_DIR)
print("REPORT_DIR =", REPORT_DIR)

## 3. GitHub repo clone

In [ ]:
%cd /content
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!ls -lh

## 3.5 Colab hotfix

GitHub에 구버전 스크립트가 남아 있어도 이 셀이 학습 스크립트를 현재 Colab 실행에 맞게 패치합니다.

In [ ]:
from pathlib import Path

path = Path("train_step4_qwen_lora.py")
text = path.read_text(encoding="utf-8")

if "--save-steps" not in text:
    text = text.replace(
        '  parser.add_argument("--early-stopping-patience", type=int, default=3)\n',
        '  parser.add_argument("--early-stopping-patience", type=int, default=3)\n'
        '  parser.add_argument("--save-steps", type=int, default=100)\n'
        '  parser.add_argument("--eval-steps", type=int, default=100)\n'
        '  parser.add_argument("--resume-from-checkpoint", default="auto")\n',
    )

if 'use_bf16 = str(compute_dtype) == "torch.bfloat16"' not in text:
    text = text.replace(
        '  compute_dtype = select_compute_dtype()\n',
        '  compute_dtype = select_compute_dtype()\n'
        '  use_bf16 = str(compute_dtype) == "torch.bfloat16"\n'
        '  use_fp16 = str(compute_dtype) == "torch.float16"\n',
    )

old_tokenize = '''  def tokenize(batch: dict[str, list[Any]]) -> dict[str, Any]:
    texts = [format_row(json.loads(item)) for item in batch["row_json"]]
    tokenized = tokenizer(texts, truncation=True, max_length=args.max_length, padding=False)
    tokenized["labels"] = [ids[:] for ids in tokenized["input_ids"]]
    return tokenized
'''
new_tokenize = '''  def tokenize(batch: dict[str, list[Any]]) -> dict[str, Any]:
    texts = [format_row(json.loads(item)) for item in batch["row_json"]]
    return tokenizer(texts, truncation=True, max_length=args.max_length, padding=False)
'''
text = text.replace(old_tokenize, new_tokenize)

if "resume_from_checkpoint = args.resume_from_checkpoint" not in text:
    text = text.replace(
        "  model = get_peft_model(model, lora_config)\n\n  training_args = TrainingArguments(\n",
        "  model = get_peft_model(model, lora_config)\n"
        "  resume_from_checkpoint = args.resume_from_checkpoint\n"
        "  if resume_from_checkpoint == \"auto\":\n"
        "    checkpoints = sorted(\n"
        "      Path(args.output_dir).glob(\"checkpoint-*\"),\n"
        "      key=lambda path: int(path.name.rsplit(\"-\", 1)[-1]) if path.name.rsplit(\"-\", 1)[-1].isdigit() else -1,\n"
        "    )\n"
        "    resume_from_checkpoint = str(checkpoints[-1]) if checkpoints else None\n"
        "  elif resume_from_checkpoint.lower() in {\"\", \"none\", \"false\", \"0\"}:\n"
        "    resume_from_checkpoint = None\n\n"
        "  training_args = TrainingArguments(\n",
    )

text = text.replace('    eval_strategy="epoch",\n    save_strategy="epoch",\n', '    eval_strategy="steps",\n    eval_steps=args.eval_steps,\n    save_strategy="steps",\n    save_steps=args.save_steps,\n')
text = text.replace('    bf16=str(compute_dtype).endswith("bfloat16"),\n    fp16=str(compute_dtype).endswith("float16"),\n', '    bf16=use_bf16,\n    fp16=use_fp16,\n')
text = text.replace('  trainer.train()\n', '  trainer.train(resume_from_checkpoint=resume_from_checkpoint)\n')

path.write_text(text, encoding="utf-8")
print("patched train_step4_qwen_lora.py for Drive checkpoints and Colab dtype/padding")

## 4. 패키지 설치

설치 후 런타임 재시작 안내가 뜨면 재시작한 뒤 2번 셀부터 다시 실행하세요.

In [ ]:
!pip install -U pip
!pip install -e . --no-deps
!pip install -U bitsandbytes accelerate peft transformers datasets huggingface_hub
!pip install "tensorboard==2.20.0" "protobuf>=5.28.0,<6.0.0"
!pip uninstall -y torchao

## 5. 현재 repo 파일 점검

In [ ]:
from pathlib import Path
required = [
    "train_step4_qwen_lora.py",
    "evaluate_step4_sllm.py",
    "step4/train.jsonl",
    "step4/valid.jsonl",
    "step4/test.jsonl",
    "step4/label_stats.json",
]
missing = [path for path in required if not Path(path).exists()]
if missing:
    raise FileNotFoundError(missing)
print("all required files exist")

## 6. 데이터셋 확인

In [ ]:
import json
from pathlib import Path

!ls -lh step4
stats = json.loads(Path("step4/label_stats.json").read_text(encoding="utf-8"))
print(json.dumps(stats["splits"], ensure_ascii=False, indent=2))

## 7. Drive checkpoint 확인

이 셀에서 기존 checkpoint가 보이면 다음 학습은 자동으로 이어서 시작합니다.

In [ ]:
!mkdir -p {OUTPUT_DIR}
!find {OUTPUT_DIR} -maxdepth 1 -type d -name 'checkpoint-*' -print | sort || true
!ls -lh {OUTPUT_DIR} | head || true

## 8. T4용 QLoRA 학습

처음 검증용으로 `--epochs 1`만 돌립니다. 출력과 checkpoint는 Drive에 저장됩니다.

In [ ]:
!python train_step4_qwen_lora.py \
  --train-file step4/train.jsonl \
  --valid-file step4/valid.jsonl \
  --output-dir {OUTPUT_DIR} \
  --batch-size 1 \
  --gradient-accumulation-steps 16 \
  --max-length 512 \
  --epochs 1 \
  --save-steps 100 \
  --eval-steps 100 \
  --resume-from-checkpoint auto

## 9. 학습 산출물 확인

In [ ]:
!ls -lh {OUTPUT_DIR}
required_model_files = ["adapter_config.json", "adapter_model.safetensors"]
missing_model_files = [name for name in required_model_files if not Path(OUTPUT_DIR, name).exists()]
if missing_model_files:
    raise FileNotFoundError(f"training output is incomplete: {missing_model_files}")
print("adapter files exist in Drive")

## 10. 평가 실행

평가 결과도 Drive에 저장합니다.

In [ ]:
!mkdir -p {REPORT_DIR}
!python evaluate_step4_sllm.py \
  --dataset step4/test.jsonl \
  --model-path {OUTPUT_DIR} \
  --output-dir {REPORT_DIR}

## 11. 평가 결과 확인

In [ ]:
!cat {REPORT_DIR}/step4_eval_summary.json
!echo "\n--- tag-wise csv ---"
!head -n 40 {REPORT_DIR}/step4_eval_summary.csv
!echo "\n--- false positives count ---"
!wc -l {REPORT_DIR}/step4_false_positives.jsonl

## 12. Hugging Face 로그인

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 13. Hugging Face Hub 업로드

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print("uploaded to", HF_REPO_ID)

## 14. 로컬 설정에 반영할 값

업로드가 끝나면 로컬 설정의 `pii.step4.local_model_path`를 아래 값으로 사용합니다.

```yaml
pii:
  step4:
    provider: local_qwen
    local_model_path: bbanany/qwen2.5-1.5b-step4-pii-qlora
```